# **6. Stronger Models (XGBoost / LightGBM / CatBoost)**
Inputs: Day 4 three-way split (`X_train`/`X_val`/`X_test`) + Day 5 baselines & feature-selection.
Move from bagged (RF) to boosted ensembles with early stopping on `X_val`; one honest `X_test` look at the end.

**Prerequisite:** `xgboost`, `lightgbm`, `catboost` must be installed in the venv before running.

### **Overview & key decisions**
- **Evaluation partition:** all boosted models are scored on `X_val` (early stopping + eval); `X_test` is touched exactly once in Step 6.
- **Training data:** a single common **2M stratified subsample** of `X_train` is used for XGBoost / LightGBM / CatBoost (consistent, apples-to-apples, manageable runtime).
- **CatBoost representation:** run on the **same one-hot `FEATURES_DAY6` matrix** as XGBoost/LightGBM. Native-categorical reconstruction was considered, but `loan_to_income_ratio`, `applicant_age`, and the `*_missing` flags were derived in Day 4 in-memory and are not persisted in the source parquets; using the one-hot matrix keeps all six models on identical feature space (a permitted option in the plan, and the honest choice given the data).
- **Imbalance:** `scale_pos_weight = neg/pos` for XGBoost & LightGBM; `auto_class_weights='Balanced'` for CatBoost.
- **Feature set:** decided by the Step 0 ablation of Day 5's 77-keep/26-drop table (with a fix for a fully-emptied family).
- **Artifacts:** `day6_model_metrics.csv`, `day6_combined_importance.csv`, ROC/PR overlays, `markdown/day6_stronger_models_summary.md`.

In [ ]:
import numpy as np, pandas as pd, os, time
import xgboost, lightgbm, catboost
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_score,
                             recall_score, f1_score, confusion_matrix, roc_curve, precision_recall_curve)
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import os as _os, pathlib as _pl
_ROOT = None
for _cand in [_pl.Path(_os.getcwd()), _pl.Path(_os.getcwd()).parent,
              _pl.Path('/Volumes/Mitul/Projects/home-mortage-approval-predictor')]:
    if (_cand / 'data' / 'processed' / 'modelling').exists():
        _ROOT = _cand; break
if _ROOT is None:
    _ROOT = _pl.Path(_os.getcwd())
M = str(_ROOT / 'data' / 'processed' / 'modelling') + '/'
FIG = str(_ROOT / 'figures' / 'day6') + '/'
os.makedirs(FIG, exist_ok=True)

X_train = pd.read_parquet(M + 'X_train.parquet').astype(np.float32)
X_val   = pd.read_parquet(M + 'X_val.parquet').astype(np.float32)
X_test  = pd.read_parquet(M + 'X_test.parquet').astype(np.float32)
y_train = pd.read_parquet(M + 'y_train.parquet')['approved'].astype('int8').values
y_val   = pd.read_parquet(M + 'y_val.parquet')['approved'].astype('int8').values
y_test  = pd.read_parquet(M + 'y_test.parquet')['approved'].astype('int8').values
val_demo = pd.read_parquet(M + 'val_demographics_lookup.parquet')
ALL_FEATURES = list(X_train.columns)  # 103

# Common 2M stratified subsample (all boosted baselines)
SUB_N = 2_000_000
Xsub_oh, _, ysub, _ = train_test_split(X_train, y_train, train_size=SUB_N, random_state=42, stratify=y_train)

spw = float((y_train == 0).sum()) / float((y_train == 1).sum())
print('scale_pos_weight (neg/pos) =', round(spw, 4))
print('libs:', xgboost.__version__, lightgbm.__version__, catboost.__version__)

def evaluate_model(model, X, y, name='model'):
    proba = model.predict_proba(X)[:, 1]
    pred = (proba >= 0.5).astype(int)
    r = dict(name=name, roc_auc=roc_auc_score(y, proba), pr_auc=average_precision_score(y, proba),
             precision=precision_score(y, pred), recall=recall_score(y, pred),
             f1=f1_score(y, pred), cm=confusion_matrix(y, pred), proba=proba)
    print(f"[{name}] ROC-AUC={r['roc_auc']:.4f} PR-AUC={r['pr_auc']:.4f} P={r['precision']:.4f} R={r['recall']:.4f} F1={r['f1']:.4f}")
    return r

results = {}
print('setup done | one-hot subsample', Xsub_oh.shape)


#### **What this cell does / output**
- Loads the Day-4 three-way split as float32 (`X_train` 5.95M×103, `X_val` 661K, `X_test` 1.65M) and `y_*` as int8.
- Builds **one common 2M stratified subsample** of `X_train` (used by every boosted model for a fair, fixed-comparison training set).
- `scale_pos_weight = 0.3396` (neg/pos ratio; approval is the ~74.7% majority class, so the positive class is weighted up ~3×).
- Prints library versions (xgboost 3.4.1, lightgbm 4.7.0, catboost 1.2.10) and defines the shared `evaluate_model` (ROC-AUC, PR-AUC, Precision/Recall/F1, confusion matrix) reused by all models.
- Output line: `setup done | one-hot subsample (2000000, 103)`.


## **Step 0 - Ablate the Notebook 5 feature-selection recommendation**
Test the '77 keep / 26 drop' table on the models it wasn't derived from before trusting it.
- **Family integrity:** confirm no one-hot family is fully emptied by the 26 drops; if one is (e.g. `negative_amortization`), force-keep one level as an implicit reference.
- **Ablation:** train a quick LightGBM on the full 103-feature set vs the trimmed set (same subsample, same params, same early stopping); if the ROC-AUC/PR-AUC gap is within ~0.002, proceed with the trimmed set for faster iteration.
- **Deliverable:** `FEATURES_DAY6` used by every model below.

In [ ]:
# Step 0a: family integrity of Day 5 drop list
fs = pd.read_csv(M + 'day5_feature_selection.csv')
drops = set(fs[fs['recommendation'].str.startswith('drop')]['feature'])
fam = {}
for c in ALL_FEATURES:
    fam.setdefault(c.split('_', 1)[0], []).append(c)
fully_emptied = [f for f, members in fam.items() if set(members).issubset(drops)]
print('families fully emptied by raw drops:', fully_emptied)
adjusted_drops = set(drops)
for f in fully_emptied:
    keep = fam[f][0]            # keep first level (implicit reference)
    adjusted_drops.discard(keep)
    print(f"  force-keep {keep} (family '{f}' would otherwise be fully dropped)")
print('adjusted drop count:', len(adjusted_drops))

FEATURES_FULL = ALL_FEATURES
FEATURES_TRIM = [c for c in ALL_FEATURES if c not in adjusted_drops]
print('FEATURES_FULL:', len(FEATURES_FULL), '| FEATURES_TRIM:', len(FEATURES_TRIM))


### **Output — family integrity fix**
- Day 5's 26-drop list would have **fully emptied the `negative` family** (every `negative_amortization_*` level dropped), removing the implicit one-hot reference level for that category.
- Fix: force-keep `negative_amortization_1` as the reference. Adjusted drop count = **25**.
- `FEATURES_FULL = 103` (all columns); `FEATURES_TRIM = 78` (after the 25 adjusted drops).


In [ ]:
def quick_lgbm(features):
    m = LGBMClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8,
                       colsample_bytree=0.8, scale_pos_weight=spw, n_jobs=-1, random_state=42,
                       early_stopping_rounds=50)
    m.fit(Xsub_oh[features].values, ysub,
          eval_set=[(Xsub_oh[features].values, ysub), (X_val[features].values, y_val)],
          eval_metric='auc')
    return m

m_full = quick_lgbm(FEATURES_FULL)
m_trim = quick_lgbm(FEATURES_TRIM)
auc_full = roc_auc_score(y_val, m_full.predict_proba(X_val[FEATURES_FULL])[:, 1])
pr_full  = average_precision_score(y_val, m_full.predict_proba(X_val[FEATURES_FULL])[:, 1])
auc_trim = roc_auc_score(y_val, m_trim.predict_proba(X_val[FEATURES_TRIM])[:, 1])
pr_trim  = average_precision_score(y_val, m_trim.predict_proba(X_val[FEATURES_TRIM])[:, 1])
print(f"FULL {len(FEATURES_FULL)}: ROC-AUC={auc_full:.4f} PR-AUC={pr_full:.4f}")
print(f"TRIM {len(FEATURES_TRIM)}: ROC-AUC={auc_trim:.4f} PR-AUC={pr_trim:.4f}")
gap = abs(auc_full - auc_trim)
FEATURES_DAY6 = FEATURES_TRIM if gap < 0.002 else FEATURES_FULL
print(f"ablation gap={gap:.4f} -> FEATURES_DAY6 = {len(FEATURES_DAY6)} features ({'TRIMMED' if FEATURES_DAY6 is FEATURES_TRIM else 'FULL'})")


#### **Output — ablation decision**
- A quick LightGBM (500 iters) is trained on the full vs the trimmed feature set and scored on `X_val`.
- FULL (103): ROC-AUC=0.8717, PR-AUC=0.9436.  TRIM (78): ROC-AUC=0.8712, PR-AUC=0.9434.
- Ablation gap = **0.0004** (< the 0.002 acceptance threshold) → adopts **`FEATURES_DAY6` = the 78-feature trimmed set** for all subsequent models (faster, virtually identical performance).


## **Step 1 - Shared setup for boosted models**
One shared `evaluate_model`, one `eval_metric='AUC'`, one `early_stopping_rounds=50` on `X_val`, and per-library imbalance handling documented in one place so every model is comparable.

In [ ]:
print('=== Shared protocol (Step 1) ===')
print('eval_metric = AUC | early_stopping_rounds = 50 | eval_set = (2M subsample train, X_val)')
print(f'XGBoost  : scale_pos_weight = {spw:.4f}')
print(f'LightGBM : scale_pos_weight = {spw:.4f}  (chosen over is_unbalance for consistency)')
print(f'CatBoost : auto_class_weights = "Balanced"')
print(f'RF bar to beat: ROC-AUC 0.8732, recall 0.9652 (high recall may be a class-weight artifact)')
print('FEATURES_DAY6:', len(FEATURES_DAY6))


### Output — shared protocol
- `eval_metric = AUC`; `early_stopping_rounds = 50` on `X_val` for every boosted model.
- Imbalance handling: XGBoost & LightGBM use `scale_pos_weight = 0.3396`; CatBoost uses `auto_class_weights = 'Balanced'`.
- RF bar to beat (Day 5): ROC-AUC 0.8732, recall 0.9652 — the very high recall is likely a class-weight artifact, noted for later.
- Confirms `FEATURES_DAY6 = 78`.


## **Step 2 - XGBoost baseline (untuned)**
`XGBClassifier` with moderate depth, `learning_rate=0.05`, many estimators (early stopping does the work), `scale_pos_weight` from Step 1. Gain-based importances extracted and compared against Day 5's RF.

In [ ]:
xgb = XGBClassifier(n_estimators=2000, max_depth=6, learning_rate=0.05, subsample=0.8,
                    colsample_bytree=0.8, scale_pos_weight=spw, importance_type='gain',
                    eval_metric='auc', random_state=42, n_jobs=-1, early_stopping_rounds=50)
Xtr = Xsub_oh[FEATURES_DAY6].values; Xva = X_val[FEATURES_DAY6].values
xgb.fit(Xtr, ysub, eval_set=[(Xtr, ysub), (Xva, y_val)], verbose=False)
print('XGBoost best_iteration:', xgb.best_iteration)
results['XGB'] = evaluate_model(xgb, Xva, y_val, 'XGBoost')
xgb_imp = pd.Series(xgb.feature_importances_, index=FEATURES_DAY6).sort_values(ascending=False).reset_index()
xgb_imp.columns = ['feature', 'gain']; xgb_imp['rank'] = np.arange(1, len(xgb_imp) + 1)
print(xgb_imp.head(15))


### Output — XGBoost (untuned)
- Trained on the 2M subsample; early-stopping did not trigger within 2000 iters (best_iteration = 1999).
- **X_val: ROC-AUC=0.8851, PR-AUC=0.9495, Precision=0.9124, Recall=0.8418, F1=0.8757** — already beats the RF bar (0.8732).
- Gain-based importances captured for the Step-5 combined table (top: loan_purpose, derived_dwelling_category, debt_to_income_ratio_missing, income, loan_to_value_ratio, loan_to_income_ratio).


## **Step 3 - LightGBM baseline (untuned)**
Comparable defaults to XGBoost for a fair head-to-head. Training time is noted explicitly - a legitimate large-scale data point.

In [ ]:
t0 = time.time()
lgbm = LGBMClassifier(n_estimators=2000, max_depth=6, learning_rate=0.05, subsample=0.8,
                     colsample_bytree=0.8, scale_pos_weight=spw, random_state=42, n_jobs=-1,
                     early_stopping_rounds=50)
lgbm.fit(Xsub_oh[FEATURES_DAY6].values, ysub,
         eval_set=[(Xsub_oh[FEATURES_DAY6].values, ysub), (X_val[FEATURES_DAY6].values, y_val)],
         eval_metric='auc')
t_lgbm = time.time() - t0
print(f'LightGBM train time: {t_lgbm:.1f}s | best_iteration: {lgbm.best_iteration_}')
results['LGBM'] = evaluate_model(lgbm, X_val[FEATURES_DAY6].values, y_val, 'LightGBM')
lgbm_imp = pd.Series(lgbm.booster_.feature_importance(importance_type='gain'),
                    index=FEATURES_DAY6).sort_values(ascending=False).reset_index()
lgbm_imp.columns = ['feature', 'gain']; lgbm_imp['rank'] = np.arange(1, len(lgbm_imp) + 1)
print(lgbm_imp.head(15))
print(f'Training-time note: LightGBM {t_lgbm:.1f}s; typically faster than XGBoost at this scale.')


#### **Output — LightGBM (untuned)**
- Train time **266.5s**, best_iteration = 2000 (no early stop within budget).
- **X_val: ROC-AUC=0.8823, PR-AUC=0.9482, Precision=0.9118, Recall=0.8363, F1=0.8724.**
- Slightly below XGBoost but clearly above RF — confirms gradient-boosted trees dominate the bagged RF at this scale.


## **Step 4 - CatBoost baseline (untuned, one-hot matrix)**
Run on the **same one-hot `FEATURES_DAY6` matrix** as XGBoost/LightGBM for a clean three-way comparison. Native-categorical reconstruction was considered but deferred: `loan_to_income_ratio`, `applicant_age`, and the `*_missing` flags were derived in Day 4 in-memory and are not persisted in the source parquets, so the one-hot matrix is used to keep all six models on identical feature space. Early stopping on `X_val`, `auto_class_weights='Balanced'`.

In [ ]:
cat = CatBoostClassifier(iterations=2000, learning_rate=0.05, depth=6,
                         auto_class_weights='Balanced', eval_metric='AUC',
                         early_stopping_rounds=50, random_state=42, verbose=False)
cat.fit(Xsub_oh[FEATURES_DAY6].values, ysub, eval_set=[(X_val[FEATURES_DAY6].values, y_val)])
print('CatBoost best_iteration:', cat.get_best_iteration())
results['CatBoost'] = evaluate_model(cat, X_val[FEATURES_DAY6].values, y_val, 'CatBoost')
cat_imp = pd.Series(cat.get_feature_importance(), index=FEATURES_DAY6).sort_values(ascending=False).reset_index()
cat_imp.columns = ['feature', 'importance']; cat_imp['rank'] = np.arange(1, len(cat_imp) + 1)
print(cat_imp.head(15))
print('CatBoost representation: one-hot FEATURES_DAY6 matrix (native-categorical reconstruction deferred - see note).')


#### **Output — CatBoost (untuned, one-hot matrix)**
- `best_iteration = 1999`.
- **X_val: ROC-AUC=0.8794, PR-AUC=0.9470, Precision=0.9112, Recall=0.8311, F1=0.8693.**
- Lowest of the three boosted models, yet still > RF. Run on the **one-hot `FEATURES_DAY6` matrix**; native-categorical reconstruction was deferred because `loan_to_income_ratio`, `applicant_age`, and the `*_missing` flags were derived in Day 4 in-memory and are not persisted — the one-hot matrix keeps all six models on identical feature space.


## **Step 5 - Four-way (+ baselines) model comparison**
- 6-model metrics table: LR / DT / RF (Day 5) + XGBoost / LightGBM / CatBoost (today), all on `X_val`.
- Overlaid ROC + PR curves (reusing Day 5's plotting pattern).
- Combined cross-model importance (XGB gain / LightGBM gain / CatBoost importance / RF impurity) on the one-hot space. RF is re-fit here (1.5M) so its proba/importance are available for the overlay and combined table, consistent with Day 5 numbers.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
Xrf, _, yrf, _ = train_test_split(X_train, y_train, train_size=1_500_000, random_state=42, stratify=y_train)
rf = RandomForestClassifier(n_estimators=150, class_weight='balanced_subsample', n_jobs=-1, random_state=42).fit(Xrf, yrf)
results['RF'] = evaluate_model(rf, X_val, y_val, 'RandomForest')
rf_imp = pd.Series(rf.feature_importances_, index=ALL_FEATURES).sort_values(ascending=False).reset_index()
rf_imp.columns = ['feature', 'impurity']; rf_imp['rank'] = np.arange(1, len(rf_imp) + 1)

base = pd.read_csv(M + 'day5_baseline_metrics.csv', index_col=0)
metrics = pd.concat([base, pd.DataFrame({k: results[k] for k in ['XGB', 'LGBM', 'CatBoost']},
                          index=['roc_auc', 'pr_auc', 'precision', 'recall', 'f1']).T])
print(metrics.round(4))

plt.figure(figsize=(6, 6))
for name, r in results.items():
    fpr, tpr, _ = roc_curve(y_val, r['proba']); plt.plot(fpr, tpr, label=f"{name} (AUC={r['roc_auc']:.3f})")
plt.plot([0, 1], [0, 1], 'k--'); plt.xlabel('FPR'); plt.ylabel('TPR'); plt.legend()
plt.title('ROC - Day6 models (X_val)'); plt.savefig(FIG + 'roc_overlay.png', dpi=120); plt.close()
plt.figure(figsize=(6, 6))
for name, r in results.items():
    prec, rec_, _ = precision_recall_curve(y_val, r['proba']); plt.plot(rec_, prec, label=f"{name} (AP={r['pr_auc']:.3f})")
plt.xlabel('Recall'); plt.ylabel('Precision'); plt.legend()
plt.title('PR - Day6 models (X_val)'); plt.savefig(FIG + 'pr_overlay.png', dpi=120); plt.close()

combo = pd.DataFrame({'XGB_gain_rank': xgb_imp.set_index('feature')['rank'],
                      'LGBM_gain_rank': lgbm_imp.set_index('feature')['rank'],
                      'CatBoost_rank': cat_imp.set_index('feature')['rank'],
                      'RF_rank': rf_imp.set_index('feature')['rank']}).fillna(len(ALL_FEATURES))
combo['avg_rank'] = combo.mean(axis=1); combo = combo.sort_values('avg_rank')
print('Combined cross-model importance (one-hot space), top 15:')
print(combo.head(15))
combo.to_csv(M + 'day6_combined_importance.csv', index=True)
metrics.to_csv(M + 'day6_model_metrics.csv')
print('saved overlays + metrics + combined importance')


#### **Output — 6-model comparison**
- Re-fits a 1.5M-row RF (X_val ROC-AUC=0.8732, PR-AUC=0.9417, P=0.8569, R=0.9652, F1=0.9078) so its probabilities/importance join the overlay and combined table.
- Metrics table (LR/DT/RF from Day 5 + XGBoost/LightGBM/CatBoost) saved to `day6_model_metrics.csv`.
- ROC & PR overlays saved to `figures/day6/roc_overlay.png` and `pr_overlay.png`.
- Combined cross-model importance (XGB gain / LightGBM gain / CatBoost / RF impurity) on the one-hot space saved to `day6_combined_importance.csv`. **Top drivers by average rank:** `loan_purpose_1`, `debt_to_income_ratio_missing`, `income`, `derived_dwelling_category_Single Family (1-4 Units):Manufactured`, `loan_to_value_ratio`, `loan_to_income_ratio`. The race-proxy `tract_minority_population_percent` ranks mid-pack (~20) — not a dominant driver.


## **Step 6 - One honest look at `X_test`**
`X_test` has been reserved since notebook 4/5. Take the single best model (by `X_val` ROC-AUC) and evaluate it **exactly once** on `X_test`; report the val-vs-test gap. Do NOT re-tune based on this number.

In [ ]:
best = max(results, key=lambda k: results[k]['roc_auc'])
print('Best model on X_val by ROC-AUC:', best, f"({results[best]['roc_auc']:.4f})")
if best == 'RF':
    res_test = evaluate_model(rf, X_test, y_test, 'RandomForest (X_test)')
else:
    mdl = {'XGB': xgb, 'LGBM': lgbm, 'CatBoost': cat}[best]
    res_test = evaluate_model(mdl, X_test[FEATURES_DAY6].values, y_test, f"{best} (X_test)")
val_auc = results[best]['roc_auc']; test_auc = res_test['roc_auc']
print(f"VAL ROC-AUC={val_auc:.4f} vs TEST ROC-AUC={test_auc:.4f} | gap={test_auc - val_auc:+.4f}")
print('NOTE: single, un-iterated test look. No re-tuning based on this number.')


#### **Output — single honest X_test look**
- Best model on `X_val` = **XGBoost (0.8851)**.
- **X_test: ROC-AUC=0.8854, PR-AUC=0.9495, Precision=0.9125, Recall=0.8421, F1=0.8759.**
- **VAL vs TEST ROC-AUC gap = +0.0002** — essentially identical; a strong signal of clean generalization with no leakage. This look is taken exactly once and is not used to re-tune.


## **Step 7 - Wrap-up**
Writes `markdown/day6_stronger_models_summary.md` and surfaces open questions for Day 7 (Optuna hyperparameter priorities, subsampling budget for tuning, and whether the winning model's importance disagrees with Day 5's RF enough to change Day 8's SHAP focus).

In [ ]:
summary_lines = [
 "# Day 6 Stronger Models - Summary",
 "",
 f"- Feature ablation (Step 0): full {len(FEATURES_FULL)} vs trimmed {len(FEATURES_TRIM)} (adjusted: force-kept one level of fully-emptied family {fully_emptied}). Gap={gap:.4f} -> FEATURES_DAY6 = {len(FEATURES_DAY6)} ({'trimmed' if FEATURES_DAY6 is FEATURES_TRIM else 'full'}).",
 f"- Imbalance: XGB/LGBM scale_pos_weight={spw:.4f}; CatBoost auto_class_weights=Balanced. Early stopping=50 on X_val (AUC).",
 "- CatBoost: run on one-hot FEATURES_DAY6 (native-categorical reconstruction deferred - LTI/applicant_age/*_missing derived in Day 4, not persisted).",
 "- X_val metrics:",
 metrics.round(4).to_string(),
 "",
 f"- Best model on X_val: {best} (ROC-AUC {results[best]['roc_auc']:.4f}).",
 f"- Single X_test look ({best}): ROC-AUC={res_test['roc_auc']:.4f}, PR-AUC={res_test['pr_auc']:.4f}, P={res_test['precision']:.4f}, R={res_test['recall']:.4f}, F1={res_test['f1']:.4f}.",
 f"- Val vs Test ROC-AUC gap: {test_auc - val_auc:+.4f}.",
 "- Open questions for Day 7:",
 "  1. Which hyperparameters need Optuna: depth/leaves, lr, reg (reg_alpha/lambda, min_child_weight), subsample/colsample?",
 "  2. Subsampling strategy for Optuna at this row count (budget)?",
 f"  3. Does {best} importance disagree with Day5 RF in a way that changes Day8 SHAP focus?",
]
summary = "\n".join(summary_lines)
print(summary)
open('/Volumes/Mitul/Projects/home-mortage-approval-predictor/markdown/day6_stronger_models_summary.md', 'w').write(summary)
print('wrote markdown/day6_stronger_models_summary.md')


### Output — written summary
- Writes `markdown/day6_stronger_models_summary.md` (the narrative file) and prints it. Surfaces Day 7 open questions: which hyperparameters to tune via Optuna, the subsampling budget for tuning, and whether the winning model's importance differs enough from Day 5's RF to shift Day 8 SHAP focus.


## Day 6 - Results summary

All cells executed end-to-end with no errors. Outcomes:

**Feature set (Step 0):** Day 5's 77-keep/26-drop table would have fully emptied the `negative_amortization` family; force-kept `negative_amortization_1`. Ablation gap full (103) vs trimmed (78) = 0.0004 → adopted **`FEATURES_DAY6` = 78 trimmed features**.

**Boosted models on X_val** (early stopping = 50; XGB/LightGBM `scale_pos_weight=0.3396`, CatBoost `auto_class_weights='Balanced'`):

| Model | ROC-AUC | PR-AUC | Precision | Recall | F1 |
|---|---|---|---|---|---|
| XGBoost | **0.8851** | 0.9495 | 0.9124 | 0.8418 | 0.8757 |
| LightGBM | 0.8823 | 0.9482 | 0.9118 | 0.8363 | 0.8724 |
| CatBoost | 0.8794 | 0.9470 | 0.9112 | 0.8311 | 0.8693 |
| RF (Day5 / refit) | 0.8732 | 0.9417 | 0.8569 | 0.9652 | 0.9078 |

All three boosted ensembles beat the RF baseline; **XGBoost is the strongest.**

**Single honest X_test look (XGBoost):** ROC-AUC **0.8854**, PR-AUC 0.9495, P=0.9125, R=0.8421, F1=0.8759. Val↔Test gap **+0.0002** (clean generalization, no leakage signal).

**Top drivers** (combined cross-model importance, one-hot space): `loan_purpose`, `debt_to_income_ratio_missing`, `income`, `derived_dwelling_category` (Single Family/Manufactured), `loan_to_value_ratio`, `loan_to_income_ratio`. The race-proxy `tract_minority_population_percent` is a mid-rank (~20) driver, not dominant.

**Artifacts:** `day6_model_metrics.csv`, `day6_combined_importance.csv`, `figures/day6/roc_overlay.png`, `figures/day6/pr_overlay.png`, `markdown/day6_stronger_models_summary.md`.

**Next (Day 7):** Optuna hyperparameter tuning on the winning XGBoost (and LightGBM); decide the tuning subsample budget; then Day 8 SHAP, Day 9 fairness, Day 10 deployment.

**Decisions documented:** CatBoost run on the one-hot matrix (native-categorical reconstruction deferred — `loan_to_income_ratio`, `applicant_age`, and the `*_missing` flags were derived in Day 4 in-memory and are not persisted). All three boosted models train/predict on positional numpy arrays to avoid library-specific feature-name restrictions (LightGBM 4.7 / XGBoost 3.x reject special characters in one-hot labels).
